In [ ]:
from datetime import datetime, timedelta
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.stats import boxcox
from tqdm import tqdm


import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, Dataset

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

## LSTNet

In [ ]:
class LSTConvNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.conv1d = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            padding=kernel_size - 1,
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        out = self.relu(self.conv1d(X))
        out = out[:, :, :-(self.kernel_size - 1)].contiguous()
        out = self.dropout(out)
        return out
    

class LSTGruNet(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        _, hidden = self.gru(X)
        return self.dropout(hidden[-1])  # [batch_size, hidden_size]


class LSTSkipGruNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        skip_step_sizes: list[int],
        skip_out_channels: list[int],
        dropout: float = 0.2,
    ):
        super().__init__()
        assert len(skip_step_sizes) == len(skip_out_channels)
        assert len(skip_step_sizes) > 0
        self.rnn_skip_step_sizes = skip_step_sizes
        self.rnn_skip_out_channels = skip_out_channels
        self.rnn_skip_nets = nn.ModuleList()
        for i in range(len(self.rnn_skip_step_sizes)):
            skip_net = nn.GRU(
                input_size=in_channels,
                hidden_size=skip_out_channels[i],
                batch_first=True
            )
            self.rnn_skip_nets.append(skip_net)
        self.skip_dropout = nn.Dropout(dropout)

    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        skip_outputs: list[torch.Tensor] = []
        
        batch_size, _, n_timesteps = X.size()
        for i, skip_step_size in enumerate(self.rnn_skip_step_sizes):
            n_skip_sequences = n_timesteps // skip_step_size
            skip_in = X[:, :, -n_skip_sequences*skip_step_size:].contiguous()
            skip_in = skip_in.view(skip_in.size(0), skip_in.size(1), n_skip_sequences, skip_step_size)
            skip_in = skip_in.permute(0, 3, 2, 1).contiguous()
            skip_in = skip_in.view(skip_in.size(0) * skip_in.size(1), skip_in.size(2), skip_in.size(3))
            
            _, skip_hidden = self.rnn_skip_nets[i](skip_in)
            skip_out = skip_hidden[-1]
    
            skip_out = skip_out.view(batch_size, skip_step_size * skip_out.size(1))
            skip_out = self.skip_dropout(skip_out)
            skip_outputs.append(skip_out)
        
        return torch.cat(skip_outputs, dim=1)


class LSTNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        horizon: int,
        cnn_out_channels: int = 32,
        cnn_kernel_size: int = 2,
        rnn_out_channels: int = 64,
        rnn_skip_step_sizes: list[int] | None = None,
        rnn_skip_out_channels: list[int] | None = None,
        ar_window_size: int = 24,
        dropout: float = 0.2
    ):
        super().__init__()

        self.conv_net = LSTConvNet(
            in_channels=in_channels,
            out_channels=cnn_out_channels,
            kernel_size=cnn_kernel_size,
            dropout=dropout,
        )
        
        # Recurrent GRU Net
        self.rnn_net = LSTGruNet(
            input_size=cnn_out_channels,
            hidden_size=rnn_out_channels,
            dropout=dropout,
        )

        # Recurrent GRU Skip Nets
        assert len(rnn_skip_step_sizes) == len(rnn_skip_out_channels)
        self.skip_rnn_net = LSTSkipGruNet(
            in_channels=cnn_out_channels,
            skip_step_sizes=rnn_skip_step_sizes,
            skip_out_channels=rnn_skip_out_channels,
            dropout=dropout
        )
        
        # Linear layer to decode rnn + skip rnn outputs
        decoder_in_channels = rnn_out_channels + np.dot(rnn_skip_step_sizes, rnn_skip_out_channels)
        self.decoder_net = nn.Linear(in_features=decoder_in_channels, out_features=horizon)
        
        # Highway Net
        self.target_index = -1
        self.ar_window_size = ar_window_size
        if self.ar_window_size > 0:
            self.ar_net = nn.Linear(self.ar_window_size, out_features=horizon)
        


    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Conv net
        conv_in = X.permute(0, 2, 1).contiguous()  # [batch_size, in_channels, time_steps] 
        conv_out = self.conv_net(conv_in)          # [batch_size, cnn_out_channels, time_steps]

        # Recurrent GRU Net
        gru_in = conv_out.permute(0, 2, 1)       # [batch_size, time_steps, cnn_out_channels]
        gru_out = self.rnn_net(gru_in)           # [batch_size, rnn_out_channels]

        # Recurrent skip GRUs
        skip_out = self.skip_rnn_net(conv_out)           # [batch_size, dot(rnn_skip_step_sizes, rnn_skip_out_channels)]
        decoder_in = torch.cat((gru_out, skip_out), 1)
        
        # Output decoder layer
        decoder_out = self.decoder_net(decoder_in)       # [batch_size, out_channels]
        
        # Ar net
        if self.ar_window_size > 0:
            ar_in = X[:, -self.ar_window_size:, [self.target_index]]  # [batch_size, ar_window, 1]
            ar_in = ar_in.permute(0, 2, 1)                               # [batch_size, 1, ar_window]
            ar_out = self.ar_net(ar_in).squeeze(1)                        # [batch_size, out_channels]
            decoder_out = decoder_out + ar_out

        return decoder_out

## PJM Dataset

In [ ]:
local_input_path = Path("../../data/pjm")
local_output_path = Path("../../results/pjm")

kaggle_input_path = Path("/kaggle/input/uci-electricity-load-2011-2014")
kaggle_output_path = Path("/kaggle/working")

INPUT_PATH = local_input_path
OUTPUT_PATH = local_output_path

### Data loading and transform utils

In [ ]:
def compute_features(df: pl.DataFrame):
    return (
        df
        .with_columns(
            hour_of_day=pl.col("timestamp").dt.hour(),
            sin_hour_of_day=(pl.col("timestamp").dt.hour() * 2 * np.pi / 24).sin(),
            cos_hour_of_day=(pl.col("timestamp").dt.hour() * 2 * np.pi / 24).cos(),

            day_of_week=pl.col("timestamp").dt.weekday(),
            sin_day_of_week=(pl.col("timestamp").dt.weekday() * 2 * np.pi / 7).sin(),
            cos_day_of_week=(pl.col("timestamp").dt.weekday() * 2 * np.pi / 7).cos(),
            is_weekend=(pl.col("timestamp").dt.weekday() >= 6).cast(pl.Float64),
            
            month_of_year=pl.col("timestamp").dt.month(),
            sin_month_of_year=(pl.col("timestamp").dt.month() * 2 * np.pi / 12).sin(),
            cos_month_of_year=(pl.col("timestamp").dt.month() * 2 * np.pi / 12).cos(),
        )
    )


class StandardScaler:
    def __init__(self):
        self._mean: float | None = None
        self._std: float | None = None

    @property
    def is_fit(self):
        return self._mean is not None and self._std is not None
    
    def fit_transform(self, y: pl.Series) -> pl.Series:
        self._mean, self._std = y.mean(), y.std()
        return self.transform(y)
    
    def transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit
        return (y - self._mean) / self._std

    def inverse_transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit
        return y * self._std + self._mean
    

class BoxCoxScaler:
    def __init__(self):
        self._lambda: float | None = None

    @property
    def is_fit(self):
        return self._lambda is not None
    
    def fit_transform(self, y: pl.Series) -> pl.Series:
        y_t, _lambda = boxcox(y.to_numpy())
        self._lambda = _lambda
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit
        y_t = boxcox(y.to_numpy(), lmbda=self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def inverse_transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit

        if self._lambda == 0:
            y_t = np.exp(y.to_numpy())
        else:
            y_t = (y.to_numpy() * self._lambda + 1) ** (1 / self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    

class TimeseriesDataset(Dataset):
    def __init__(self, timeseries: np.ndarray, input_seq_length: int, output_seq_length: int, target_col_idx: int = 0):
        super().__init__()
        if timeseries.ndim < 2:
            timeseries = timeseries.reshape(-1, 1)
        elif timeseries.ndim > 2:
            raise ValueError("Expecting input array with at most two dimensions.")
        
        self.timeseries = torch.tensor(timeseries, dtype=torch.float32)
        self.in_seq_length = input_seq_length
        self.out_seq_length = output_seq_length
        self.target_col_index = target_col_idx

    def __len__(self):
        return self.timeseries.size(0) - self.in_seq_length - self.out_seq_length

    def __getitem__(self, index) -> tuple[torch.Tensor, torch.Tensor]:
        x_start, x_end = int(index), int(index + self.in_seq_length)
        x = self.timeseries[x_start: x_end]
        
        y_start, y_end = int(x_end), int(x_end + self.out_seq_length)
        y = self.timeseries[y_start: y_end, [self.target_col_index]]
        return x, y

### Model training and evaluation utils

In [ ]:
def train_lstnet_model(model: LSTNet, dataloader: DataLoader, n_epochs: int = 50):
    model = model.to(DEVICE)

    loss_fn = nn.MSELoss()
    optimizer = AdamW(model.parameters(), lr=1e-03)

    model.train()

    all_batch_losses = []
    
    for epoch in tqdm(range(n_epochs)):
        current_epoch_batch_losses = []
        for batch_idx, (batch_X, batch_y) in enumerate(dataloader):
            
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            
            y_hat = model(batch_X)

            # Output dimension is [batch_size, out_seq_length]
            # Reshape to [batch_size, out_seq_length, 1]
            y_hat = y_hat.unsqueeze(-1)
            
            loss = loss_fn(batch_y, y_hat)
            loss.backward()
            optimizer.step()
            
            loss_detach = float(loss.detach())
            current_epoch_batch_losses.append(loss_detach)
    
        all_batch_losses.append(current_epoch_batch_losses)
    epoch_losses = np.mean(all_batch_losses, axis=1).tolist()
    metadata = {"batch_losses": all_batch_losses, "epoch_losses": epoch_losses}
    return model, metadata


def predict_lstnet_model(model: LSTNet, X_test: torch.Tensor) -> torch.Tensor:
    model.eval()
    with torch.no_grad():
        X_test = X_test.to(DEVICE)
        y_hat = model(X_test)
        y_hat = y_hat.squeeze()
    return y_hat

### Configure

In [ ]:
PJM_SITE_NAME = "NI"
PJM_DATA_FREQUENCY = "1h"

FEATURES = [
    "sin_hour_of_day",
    "cos_hour_of_day",
    "sin_day_of_week",
    "cos_day_of_week",
    "is_weekend",
    "sin_month_of_year",
    "cos_month_of_year",
]
LABEL = "y_scaled"
INPUT_SEQUENCE_LENGTH = 5 * 24
OUTPUT_SEQUENCE_LENGTH = 2 * 24
FINE_TUNE_WINDOW = timedelta(days=3 * 30)

VALIDATION_WINDOWS = [
    (datetime(2010, 6, 10, 0, 0, 0), datetime(2010, 6, 12, 0, 0, 0)),
    (datetime(2010, 7, 15, 0, 0, 0), datetime(2010, 7, 17, 0, 0, 0)),
]

### Load Dataset

In [ ]:
data_file_name = f"{PJM_SITE_NAME}_hourly.csv"
data_file_path = INPUT_PATH / data_file_name
SITE_DF = pl.read_csv(
    data_file_path,
    columns=["Datetime", f"{PJM_SITE_NAME}_MW"],
    schema={"Datetime": pl.Datetime, f"{PJM_SITE_NAME}_MW": pl.Float64},
    new_columns=["timestamp"]
)
SITE_DF = SITE_DF.sort(by="timestamp")

### Train base model

In [ ]:
MODEL = LSTNet(
    in_channels=len(FEATURES + [LABEL]),
    horizon=OUTPUT_SEQUENCE_LENGTH,
    cnn_out_channels=32,
    cnn_kernel_size=2,
    rnn_out_channels=64,
    rnn_skip_step_sizes=[24],
    rnn_skip_out_channels=[10],
    ar_window_size=24,
    dropout=0.3,
)

In [ ]:
val_starts, _ = zip(*VALIDATION_WINDOWS)
val_start_min = min(val_starts)

# 1. TRAIN TEST SPLIT
pre_train_df = SITE_DF.filter(pl.col("timestamp").lt(val_start_min))

# Scale targets  and compute features
std_scaler, bc_scaler = StandardScaler(), BoxCoxScaler()

y = pre_train_df[f"{PJM_SITE_NAME}_MW"]
y_scaled = std_scaler.fit_transform(bc_scaler.fit_transform(y))
pre_train_df = pre_train_df.with_columns(y_scaled=y_scaled)
pre_train_df = compute_features(pre_train_df)

# Prepare dataloaders
pre_train_df = pre_train_df.sort(by="timestamp")[[LABEL] + FEATURES]
pre_train_ds = TimeseriesDataset(
    pre_train_df.to_numpy(),
    input_seq_length=INPUT_SEQUENCE_LENGTH,
    output_seq_length=OUTPUT_SEQUENCE_LENGTH,
    target_col_idx=0
)
pre_train_dl = DataLoader(pre_train_ds, batch_size=32, shuffle=True)

# Train base model
MODEL, pre_train_losses = train_lstnet_model(MODEL, pre_train_dl)


epoch_losses_path = f"{kaggle_output_path}/epoch_train_loss_{PJM_SITE_NAME}.json"
with open(epoch_losses_path, "w") as fp:
    json.dump(pre_train_losses["epoch_losses"], fp)

batch_losses_path = f"{kaggle_output_path}/batch_train_loss_{PJM_SITE_NAME}.json"
with open(batch_losses_path, "w") as fp:
    json.dump(pre_train_losses["batch_losses"], fp)

In [ ]:
# Plot losses and save
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))

ax[0].plot(sum(pre_train_losses["batch_losses"], []))
ax[0].grid(which="major", lw=0.5, color="grey", ls="--", alpha=0.75)
ax[0].set(xlabel="batch", title="Batch Losses", ylabel="MSE Loss")

ax[1].plot(pre_train_losses["epoch_losses"])
ax[1].grid(which="major", lw=0.5, color="grey", ls="--", alpha=0.75)
ax[1].set(xlabel="epoch", title="Epoch Losses")

fig.tight_layout()

fig_path = f"{kaggle_output_path}/loss_curves_{PJM_SITE_NAME}.png"
plt.savefig(fig_path, dpi=300);

### Train Fine-Tuned Models and Validate

In [ ]:
for val_idx, (val_start, val_end) in enumerate(VALIDATION_WINDOWS):
    fine_tune_start = val_start - FINE_TUNE_WINDOW
    fine_tune_df = SITE_DF.filter(pl.col("timestamp").is_between(fine_tune_start, val_start, closed="left"))
    val_df = SITE_DF.filter(pl.col("timestamp").is_between(val_start, val_end, closed="left"))
    
    # Scale targets and calculate features
    std_scaler, bc_scaler = StandardScaler(), BoxCoxScaler()
    y = fine_tune_df[f"{PJM_SITE_NAME}_MW"]
    y_scaled = std_scaler.fit_transform(bc_scaler.fit_transform(y))
    fine_tune_df = fine_tune_df.with_columns(y_scaled=y_scaled)
    fine_tune_df = compute_features(fine_tune_df)

    # Construct fine tune data loaders
    fine_tune_df = fine_tune_df.sort(by="timestamp")[[LABEL] + FEATURES]
    fine_tune_ds = TimeseriesDataset(
        fine_tune_df.to_numpy(),
        input_seq_length=INPUT_SEQUENCE_LENGTH,
        output_seq_length=OUTPUT_SEQUENCE_LENGTH,
        target_col_idx=0
    )
    fine_tune_dl = DataLoader(fine_tune_ds, batch_size=32, shuffle=False)

    # Fine tune model
    # MODEL, pre_train_losses = train_lstnet_model(MODEL, fine_tune_dl, n_epochs=10)

    # Predict over validation period.
    X_test = fine_tune_df[-INPUT_SEQUENCE_LENGTH:].to_torch(dtype=pl.Float32)
    X_test = X_test.unsqueeze(0)
    y_hat = predict_lstnet_model(MODEL, X_test)
    y_hat_scaled = pl.Series(name=f"{PJM_SITE_NAME}_MW_FORECAST", values=y_hat.numpy(), dtype=pl.Float32)
    y_hat_rescaled = bc_scaler.inverse_transform(std_scaler.inverse_transform(y_hat_scaled))

    # Add to forecasts and save
    forecast_df = val_df.with_columns(y_hat_rescaled)

    # Plot forecasts and save
    
    